# Link Prediction in Social Networks
## Notebook 02: Topological Heuristics, Node2Vec & Feature Engineering

### Objectives
1. Formulate and compute topological graph heuristics (Common Neighbors, Jaccard, Adamic-Adar, Resource Allocation, Preferential Attachment).
2. Train Node2Vec random walk embeddings.
3. Perform leak-free edge dataset partitioning (Spanning Tree Preservation) and negative sampling.

In [ ]:
import sys
sys.path.append('..')

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_graph, split_edges_leak_free
from src.heuristics import common_neighbors, jaccard_coefficient, adamic_adar_index, resource_allocation_index, preferential_attachment
from src.pipeline import LinkPredictionPipeline

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

### 1. Load Graph and Perform Leak-Free Split
Notice that the test positive edges are completely removed from $G_{train}$ to prevent data leakage during feature calculation.

In [ ]:
G = load_graph(dataset_name="facebook_ego", sample_nodes=800, data_dir="../data/raw")
splits = split_edges_leak_free(G, test_ratio=0.15, val_ratio=0.05, seed=42)

print(f"Original Graph Edges:     {splits['metadata']['total_edges']:,}")
print(f"Training Graph Edges:     {splits['metadata']['train_edges']:,}")
print(f"Test Positive Edges:       {splits['metadata']['test_positive_edges']:,}")
print(f"Test Negative Edges:       {splits['metadata']['test_negative_edges']:,}")
print(f"G_train Fully Connected:  {splits['metadata']['is_train_connected']}")

### 2. Feature Extraction via Pipeline (Heuristics + Node2Vec Embeddings)

In [ ]:
pipeline = LinkPredictionPipeline(use_embeddings=True, embedding_dim=16, seed=42)
data_dict = pipeline.fit_transform_splits(splits)

print(f"Feature Count: {len(data_dict['feature_names'])}")
print("Extracted Features:", data_dict['feature_names'])

### 3. Feature Distribution Analysis (Positive Links vs Negative Links)
Observe how true links (Class 1) have significantly higher Adamic-Adar, Jaccard, and Node2Vec Cosine similarity scores than non-links (Class 0).

In [ ]:
df_train = data_dict['df_train_raw'].copy()
df_train['label'] = data_dict['y_train']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(x='label', y='adamic_adar', data=df_train, ax=axes[0], palette='Blues')
axes[0].set_title('Adamic-Adar Index by Class')
axes[0].set_xticklabels(['No Link (0)', 'Link (1)'])

sns.boxplot(x='label', y='jaccard_coefficient', data=df_train, ax=axes[1], palette='Greens')
axes[1].set_title('Jaccard Coefficient by Class')
axes[1].set_xticklabels(['No Link (0)', 'Link (1)'])

sns.boxplot(x='label', y='emb_cosine_sim', data=df_train, ax=axes[2], palette='Purples')
axes[2].set_title('Node2Vec Cosine Similarity by Class')
axes[2].set_xticklabels(['No Link (0)', 'Link (1)'])

plt.tight_layout()
plt.show()